# Gemma-Only ALFWorld Evaluation Notebook

This notebook is organized as alternating explanation-and-code blocks so each implementation chunk is documented before execution.

In this first code chunk, we implement environment setup and local-module loading. The goals are:
- Resolve the working directory robustly whether the kernel starts from the repository root or directly inside alfworld_debugger.
- Add the folder containing evaluation_single_model.py to Python import search paths.
- Prove that notebook code can call utilities from sibling .py files, including parser and helper functions that are already part of your debugger flow.

Why this matters: a notebook does not isolate you from your existing codebase. As long as the folder is importable, you can reuse the same utilities and avoid duplicating logic in cells. This keeps the notebook maintainable and consistent with your command-line implementation.

Expected result after running the next cell:
- It prints the resolved BASE_DIR.
- It confirms local imports succeeded.

In [10]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "evaluation_single_model.py").exists():
    BASE_DIR = cwd
elif (cwd / "alfworld_debugger" / "evaluation_single_model.py").exists():
    BASE_DIR = (cwd / "alfworld_debugger").resolve()
else:
    raise FileNotFoundError("Could not locate evaluation_single_model.py from current working directory.")

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from evaluation_single_model import _resolve_config_path, _extract_action_candidate
from main import parse_user_action

print(f"BASE_DIR={BASE_DIR}")
print("Local module imports succeeded.")

BASE_DIR=/mnt/raid/rl_gaming/RL4VLM_ALF/RL4VLM_conda/alfworld_debugger
Local module imports succeeded.


## Chunk 2: Parameter and Command Construction

The next code cell defines all runtime configuration for a Gemma-only evaluation run and builds the exact shell command that will be executed.

What this chunk implements in detail:
- Model endpoint parameters: model name, base URL, episode and step limits, and token budget.
- Runtime mode toggles: text-only inference and no image saving, matching your recent debugging preference to skip image handling entirely.
- Headless compatibility: optional xvfb-run wrapping so ALFWorld/AI2-THOR can run in environments without a physical display.
- Output management: creates a stable output directory for metrics and optional trajectory artifacts produced by evaluation_single_model.py.

This split is intentional: by building the command first and printing it, you can inspect and tweak parameters before actually launching a potentially long run.

Expected result after running the next cell:
- A fully assembled command list in cmd.
- A printed shell-safe representation of the exact command.

In [11]:
import shlex

MODEL = "google/gemma-3-27b-it"
BASE_URL = "http://localhost:8001/v1"
NUM_EPISODES = 2
MAX_STEPS = 3
MAX_TOKENS = 128
TEXT_ONLY = True
NO_SAVE_IMAGES = True
USE_XVFB = True

OUTPUT_DIR = BASE_DIR / "debug_outputs" / "model_debug"
CONFIG_PATH = _resolve_config_path(str(BASE_DIR / ".." / "VLM_PPO_ALF" / "alf-config.yaml"))

assert CONFIG_PATH.exists(), f"Config not found: {CONFIG_PATH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "conda", "run", "-n", "rl4vlm-alf", "python", "evaluation_single_model.py",
    "--model", MODEL,
    "--base-url", BASE_URL,
    "--config", str(CONFIG_PATH),
    "--num-episodes", str(NUM_EPISODES),
    "--max-steps", str(MAX_STEPS),
    "--max-tokens", str(MAX_TOKENS),
    "--output-dir", str(OUTPUT_DIR),
]

if TEXT_ONLY:
    cmd.append("--text-only")
if NO_SAVE_IMAGES:
    cmd.append("--no-save-images")
if USE_XVFB:
    cmd = ["xvfb-run", "-a"] + cmd

print("Running command:")
print(" ".join(shlex.quote(token) for token in cmd))

Running command:
xvfb-run -a conda run -n rl4vlm-alf python evaluation_single_model.py --model google/gemma-3-27b-it --base-url http://localhost:8001/v1 --config /mnt/raid/rl_gaming/RL4VLM_ALF/RL4VLM_conda/alfworld_debugger/../VLM_PPO_ALF/alf-config.yaml --num-episodes 2 --max-steps 3 --max-tokens 128 --output-dir /mnt/raid/rl_gaming/RL4VLM_ALF/RL4VLM_conda/alfworld_debugger/debug_outputs/model_debug --text-only --no-save-images


## Chunk 3: Execute Gemma Evaluation and Capture Output

The next code block launches the assembled command and captures both stdout and stderr directly into notebook variables so you can inspect everything in one place.

Implementation details:
- Uses subprocess.run with capture_output=True to collect full process logs without external files.
- Executes from BASE_DIR so relative paths in evaluation_single_model.py remain valid.
- Prints stdout always, stderr only when present, and raises a Python exception if return code is non-zero.

Why this approach is useful:
- It keeps the notebook self-contained and reproducible.
- It lets later cells parse textual output (for metrics path, status signals, etc.) without re-running the job.

Expected result after running the next cell:
- proc.stdout contains model outputs and evaluation traces.
- The cell stops with a clear RuntimeError if the command fails.

In [13]:
import subprocess

proc = subprocess.run(
    cmd,
    cwd=str(BASE_DIR),
    text=True,
    capture_output=True,
)

print("===== STDOUT =====")
print(proc.stdout)

if proc.stderr.strip():
    print("===== STDERR =====")
    print(proc.stderr)

if proc.returncode != 0:
    raise RuntimeError(f"Gemma evaluation failed with code {proc.returncode}")

print("Gemma evaluation finished successfully.")

===== STDOUT =====
Initialize AlfredThorEnv...
Initialize AlfredThorEnv...
Overall we have 187 games...
Evaluating with 187 games
Overall we have 187 games...
Evaluating with 187 games
Found path: /home/rl_gaming/.ai2thor/releases/thor-201909061227-Linux64/thor-201909061227-Linux64
Mono path[0] = '/home/rl_gaming/.ai2thor/releases/thor-201909061227-Linux64/thor-201909061227-Linux64_Data/Managed'
Mono config path = '/home/rl_gaming/.ai2thor/releases/thor-201909061227-Linux64/thor-201909061227-Linux64_Data/Mono/etc'
Preloaded 'ScreenSelector.so'
Display 0 'screen': 1280x1024 (primary device).
Logging to /home/rl_gaming/.config/unity3d/Allen Institute for Artificial Intelligence/AI2-Thor/Player.log
ThorEnv started.
ALFWorld model-driven debugger
Model      : google/gemma-3-27b-it
Base URL   : http://localhost:8001/v1
Config     : /mnt/raid/rl_gaming/RL4VLM_ALF/RL4VLM_conda/alfworld_debugger/../VLM_PPO_ALF/alf-config.yaml
Episodes   : 2
Max steps  : 3
Vision I/O : False
Resetting ThorEnv
T

## Chunk 4: Parse Metrics and Print a Compact Run Summary

The final code cell turns raw command output into structured run insight.

What it implements:
- Searches stdout for the emitted metrics file path.
- If path is not found, falls back to listing recent JSON artifacts in the output directory.
- Loads metrics JSON and prints a compact summary that includes model identity, step count, success rate, mean reward, and mean endpoint latency.

Why this final stage is important:
- It closes the loop from execution to measurable evaluation statistics.
- It provides quick comparability across notebook runs when you tune parameters.

Expected result after running the next cell:
- A clean JSON summary is printed for immediate interpretation.

In [ ]:
import json
import re

match = re.findall(r"Metrics path:\s*(.+)", proc.stdout)
if not match:
    print("Metrics path not found in output. Recent JSON files in output directory:")
    candidates = sorted(OUTPUT_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    for path in candidates[:5]:
        print(path)
else:
    metrics_path = Path(match[-1].strip())
    if not metrics_path.is_absolute():
        metrics_path = (BASE_DIR / metrics_path).resolve()

    print(f"Loading metrics from: {metrics_path}")
    with metrics_path.open("r", encoding="utf-8") as reader:
        metrics = json.load(reader)

    summary = {
        "model": metrics.get("model"),
        "num_episodes": metrics.get("num_episodes"),
        "total_steps": metrics.get("total_steps"),
        "episode_success_rate": metrics.get("episode_success_rate"),
        "reward_mean": metrics.get("episode_reward", {}).get("mean"),
        "goal_condition_final_mean": metrics.get("goal_condition_success_rate", {}).get("final_mean"),
        "goal_condition_max_mean": metrics.get("goal_condition_success_rate", {}).get("max_mean"),
        "image_available_steps": metrics.get("image_input", {}).get("steps_with_image_available"),
        "image_sent_steps": metrics.get("image_input", {}).get("steps_with_image_sent"),
        "latency_mean_sec": metrics.get("inference_latency_sec", {}).get("mean"),
    }
    print(json.dumps(summary, indent=2))

Loading metrics from: /mnt/raid/rl_gaming/RL4VLM_ALF/RL4VLM_conda/alfworld_debugger/debug_outputs/model_debug/gemma_3_27b_it_alf_debug_20260330_031404.json
{
  "model": "google/gemma-3-27b-it",
  "num_episodes": 1,
  "total_steps": 3,
  "episode_success_rate": 0.0,
  "reward_mean": 0.0,
  "latency_mean_sec": 1.1101760069529216
}


## Chunk 5: Explicitly Surface the Fourth Step Tuple Element (`info`)

This patch mirrors the Python change that now prints `STEP INFO STRUCTURE` after each environment step. The important idea is:

- `STEP OUTPUT STRUCTURE` summarizes the 4-item tuple `(observation, scores, dones, info)`.
- Its preview intentionally shows only the first few nested items for readability.
- The fourth element (`info`) can be missed in that preview when nested content is large.

To avoid ambiguity, the evaluator now prints a second block named `STEP INFO STRUCTURE` directly from `step_result.info`.

In the next code cell, we parse the already captured `proc.stdout` and print only the `STEP INFO STRUCTURE` sections so you can inspect keys and values clearly (admissible commands, won flags, goal success rate, game metadata, etc.) without scrolling through the full run log.

In [20]:
import re

# Extract and print only the explicit per-step info blocks.
# This reflects the Python patch that now prints "STEP INFO STRUCTURE" each step.
info_blocks = re.findall(
    r"STEP INFO STRUCTURE:\n(\{.*?\})\n\nTRANSITION:",
    proc.stdout,
    flags=re.DOTALL,
)

if not info_blocks:
    print("No STEP INFO STRUCTURE blocks found in proc.stdout.")
    print("Re-run the evaluation cell after pulling the latest evaluation_single_model.py changes.")
else:
    print(f"Found {len(info_blocks)} STEP INFO STRUCTURE block(s).")
    for idx, block in enumerate(info_blocks, start=1):
        print("\n" + "=" * 72)
        print(f"STEP INFO STRUCTURE BLOCK {idx}")
        print("=" * 72)
        print(block)


Found 3 STEP INFO STRUCTURE block(s).

STEP INFO STRUCTURE BLOCK 1
{
  "type": "dict",
  "keys": [
    "admissible_commands",
    "won",
    "goal_condition_success_rate",
    "extra.gamefile",
    "extra.expert_plan"
  ],
  "items": {
    "admissible_commands": {
      "type": "list",
      "len": 1,
      "preview": [
        {
          "type": "list",
          "len": 58,
          "preview": [
            "<str>",
            "<str>",
            "<str>"
          ]
        }
      ]
    },
    "won": {
      "type": "list",
      "len": 1,
      "preview": [
        {
          "type": "bool",
          "value": false
        }
      ]
    },
    "goal_condition_success_rate": {
      "type": "list",
      "len": 1,
      "preview": [
        {
          "type": "float",
          "value": 0.0
        }
      ]
    },
    "extra.gamefile": {
      "type": "list",
      "len": 1,
      "preview": [
        {
          "type": "str",
          "value": "/home/rl_gaming/.cache/alfwo

In [4]:
import argparse
import base64
import io
import json
import re
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
from openai import OpenAI
from PIL import Image

from env_wrapper import AlfWorldEnvWrapper
from logger import TrajectoryLogger
from main import parse_user_action
from renderer import DebugRenderer